## 1. Environment Setup
This cell configures Java and Python for Spark.


In [1]:
import os, sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PYSPARK_PYTHON'] = sys.executable

print('Python:', sys.version)
print('JAVA_HOME:', os.environ.get('JAVA_HOME'))


Python: 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


## 2. Create Spark Session


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('YellowTaxi') \
    .master('local[*]') \
    .getOrCreate()

spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/09 19:57:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Part A: Spark Dataframes


## Task A1- Load and Inspect

In [3]:
#Read parquet file
df = spark.read.parquet("/home/vboxuser/Big Data Systems Design/Assignment1/yellow_tripdata_2025-01.parquet")
df.show(5)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [4]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [6]:
# Show counts for: rows and columns
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 3475226
Columns: 20


In [7]:
print(df.columns)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


In [8]:
# import all functions
from pyspark.sql.functions import *

#Identify Columns of Interest
df_basic = df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "fare_amount",
    "trip_distance"
)

df_basic.show(5)

+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|fare_amount|trip_distance|
+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
| 2025-01-01 00:18:38|  2025-01-01 00:26:59|         229|         237|              1|       10.0|          1.6|
| 2025-01-01 00:32:40|  2025-01-01 00:35:13|         236|         237|              1|        5.1|          0.5|
| 2025-01-01 00:44:04|  2025-01-01 00:46:01|         141|         141|              1|        5.1|          0.6|
| 2025-01-01 00:14:27|  2025-01-01 00:20:01|         244|         244|              3|        7.2|         0.52|
| 2025-01-01 00:21:34|  2025-01-01 00:25:06|         244|         116|              3|        5.8|         0.66|
+--------------------+---------------------+------------+------------+---------------+----------

In [9]:
df_clean = df_basic

In [10]:
df_clean.printSchema()

root
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- trip_distance: double (nullable = true)



In [11]:
# Create a single row data frame showing the number of null values in each column of interest
null_counts = df_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_basic.columns])

#print full results with column names
null_counts.show(truncate=False)

[Stage 7:==============>                                            (1 + 3) / 4]

+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|fare_amount|trip_distance|
+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
|0                   |0                    |0           |0           |540149         |0          |0            |
+--------------------+---------------------+------------+------------+---------------+-----------+-------------+



In [12]:
# Sanity check that dates have no nulls- pickupdate
pickup = df.filter(col("tpep_pickup_datetime").isNull())
pickup.show(20, truncate=False)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+----

In [13]:
# Sanity check that dates have no nulls- dropoffdate
dropOff = df.filter(col("tpep_dropoff_datetime").isNull())
dropOff.show(20, truncate=False)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+----

In [14]:
# drop records where passenger_count has nulls
df_clean = df_basic.dropna(subset=["passenger_count"])
print("Rows:", df_clean.count())
print("Columns:", len(df_clean.columns))

Rows: 2935077
Columns: 7


In [15]:
# find the PuLocationID that are not a value in the data dictionary
df_clean.filter((df_clean.PULocationID < 1) & (df_clean.PULocationID > 265 )).show()

+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|fare_amount|trip_distance|
+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
+--------------------+---------------------+------------+------------+---------------+-----------+-------------+



In [16]:
# find the DOLocationID that are not a value in the data dictionary
df_clean.filter((df_clean.DOLocationID < 1) & (df_clean.DOLocationID > 265 )).show()

+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|fare_amount|trip_distance|
+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
+--------------------+---------------------+------------+------------+---------------+-----------+-------------+



In [17]:
#distinct values in trip_distance
df_clean.select("passenger_count").distinct().orderBy("passenger_count").show()

+---------------+
|passenger_count|
+---------------+
|              0|
|              1|
|              2|
|              3|
|              4|
|              5|
|              6|
|              7|
|              8|
|              9|
+---------------+



In [18]:
#check that the fare_amount does not have characters

bad_fares = df_clean.filter(
   (col("fare_amount").cast("double").isNull())
)

bad_fares.select("fare_amount").distinct().show(50, truncate=False)

+-----------+
|fare_amount|
+-----------+
+-----------+



In [19]:
#check if trip distance does not have characters

bad_trip_distance = df_clean.filter(
     (col("trip_distance").cast("double").isNull())
)

bad_trip_distance.select("trip_distance").distinct().show(50, truncate=False)

+-------------+
|trip_distance|
+-------------+
+-------------+



## Task A2: Dataframe Analysis

In [20]:
#A2.1: Average Trip Distance per day of month

AvgDayOfMonth=(df_clean
.withColumn("DayOfMonth", dayofmonth("tpep_pickup_datetime"))
.groupBy("DayOfMonth")
.agg(round(avg("trip_distance"),2).alias("AvTripDistByDayOfMonth"))
.orderBy("DayOfMonth")
)

AvgDayOfMonth.show(truncate=False)

[Stage 26:==============>                                           (1 + 3) / 4]

+----------+----------------------+
|DayOfMonth|AvTripDistByDayOfMonth|
+----------+----------------------+
|1         |3.88                  |
|2         |3.69                  |
|3         |3.38                  |
|4         |3.24                  |
|5         |3.83                  |
|6         |3.54                  |
|7         |3.08                  |
|8         |2.93                  |
|9         |3.04                  |
|10        |3.13                  |
|11        |2.94                  |
|12        |3.55                  |
|13        |3.27                  |
|14        |2.95                  |
|15        |2.96                  |
|16        |3.03                  |
|17        |3.12                  |
|18        |2.9                   |
|19        |3.1                   |
|20        |4.07                  |
+----------+----------------------+
only showing top 20 rows



In [21]:
#A2.1: Average Trip Distance per day of week

AvgDistDayOfWeek = (df_clean
.withColumn("DayOfWeek", dayofweek("tpep_pickup_datetime"))
.groupBy("DayOfWeek")
.agg(round(avg("trip_distance"),2).alias("AvgTripDistByDayOfWeek"))
.orderBy("DayOfWeek")
)

AvgDistDayOfWeek.show(truncate=False)

+---------+----------------------+
|DayOfWeek|AvgTripDistByDayOfWeek|
+---------+----------------------+
|1        |3.49                  |
|2        |3.51                  |
|3        |3.01                  |
|4        |3.16                  |
|5        |3.18                  |
|6        |3.2                   |
|7        |2.94                  |
+---------+----------------------+



In [22]:
#A2.2: Top 10 locations by number of trips

pickupCounts= (df_clean
.groupBy(col("PULocationID").alias("PickupLocation"))
.count().alias("CountByPULocation")
.orderBy("count",ascending=False)
)

pickupCounts.show(10, truncate=False)

+--------------+------+
|PickupLocation|count |
+--------------+------+
|237           |152649|
|161           |151967|
|132           |145438|
|236           |141674|
|186           |112714|
|230           |112650|
|162           |108709|
|142           |100905|
|138           |89039 |
|163           |87842 |
+--------------+------+
only showing top 10 rows



In [23]:
#A2-3: Average Fare per passenger count
farePerPassCount=(df_clean
.groupBy(col("passenger_count").alias("PassengerCount"))
.agg(round(avg("fare_amount"),2).alias("AverageFareAmount"))
.orderBy("passenger_count",ascending=True)
)

farePerPassCount.show(truncate=False)

+--------------+-----------------+
|PassengerCount|AverageFareAmount|
+--------------+-----------------+
|0             |15.49            |
|1             |17.14            |
|2             |19.19            |
|3             |18.85            |
|4             |21.1             |
|5             |16.79            |
|6             |17.24            |
|7             |73.5             |
|8             |68.64            |
|9             |91.33            |
+--------------+-----------------+



In [24]:
#A2-4: Filter trips longer than 10 miles and analyse the fare distribution
longTrips= df_clean.filter(col("trip_distance") > 10)

print("trips over 10:", longTrips.count(),"\n")

#Fare Analysis
longTrips.select(
min("fare_amount").alias("MinFare"),
round(avg("fare_amount"),2).alias("AvgFare"),
max("fare_amount").alias("MaxFare")
).show()

trips over 10: 218657 

+-------+-------+-------+
|MinFare|AvgFare|MaxFare|
+-------+-------+-------+
| -826.2|   60.8| 2450.9|
+-------+-------+-------+



## PART B- SPARK SQL

In [25]:
#B-1: Peak Pickup Hours

# Create or Replace view
df_clean.createOrReplaceTempView("taxi_trips")

spark.sql("""
select 
    hour(tpep_pickup_datetime),
    count(*) as TotalTrips
from taxi_trips
group by hour(tpep_pickup_datetime)
order by TotalTrips desc""").show()

+--------------------------+----------+
|hour(tpep_pickup_datetime)|TotalTrips|
+--------------------------+----------+
|                        17|    215352|
|                        18|    213969|
|                        16|    198895|
|                        15|    196331|
|                        14|    186445|
|                        19|    181472|
|                        13|    173190|
|                        21|    167736|
|                        20|    164156|
|                        12|    163778|
|                        11|    150220|
|                        22|    139463|
|                        10|    138751|
|                         9|    126662|
|                         8|    109694|
|                        23|     98859|
|                         7|     76933|
|                         0|     68530|
|                         1|     46089|
|                         6|     36613|
+--------------------------+----------+
only showing top 20 rows



In [26]:
#B-2: Total Revenue Per Day
spark.sql("""
select 
    date_format(to_date(tpep_pickup_datetime),'yyyy-MM-dd') as tripDate,
    round(sum(fare_amount),2) as FareAmount
from taxi_trips
group by date(tpep_pickup_datetime) 
order by tripDate desc""").show()



[Stage 47:===========================================>              (3 + 1) / 4]

+----------+----------+
|  tripDate|FareAmount|
+----------+----------+
|2025-02-01|       6.5|
|2025-01-31|1744222.99|
|2025-01-30|1895413.83|
|2025-01-29|1740402.31|
|2025-01-28|1629110.79|
|2025-01-27|1459896.87|
|2025-01-26|1477810.16|
|2025-01-25|1730502.76|
|2025-01-24|1778622.71|
|2025-01-23|1855999.17|
|2025-01-22|1693337.23|
|2025-01-21|1634178.63|
|2025-01-20|2204823.26|
|2025-01-19| 1302377.7|
|2025-01-18|1548082.29|
|2025-01-17|1735886.24|
|2025-01-16|1963349.21|
|2025-01-15|1800249.41|
|2025-01-14|1749766.39|
|2025-01-13|1546238.73|
+----------+----------+
only showing top 20 rows



In [27]:
#B-3: Top 5 busiest pickup/dropoff pairs
spark.sql("""
select 
    concat( PULocationID, '-', DOLocationID) as Pairs,
    count(PULocationID || DOLocationID) as PairCount
from taxi_trips
group by concat( PULocationID, '-', DOLocationID)
order by PairCount desc""").show(5)

[Stage 50:==============>                                           (1 + 3) / 4]

+-------+---------+
|  Pairs|PairCount|
+-------+---------+
|237-236|    23984|
|236-237|    20685|
|236-236|    17502|
|237-237|    16850|
|161-237|    10992|
+-------+---------+
only showing top 5 rows



In [28]:
#B-4: Detect Suspicious Trips (trip_distance < 1 fare_amount > 50)
spark.sql("""
select *
from taxi_trips
where trip_distance < 1
and fare_amount > 50""").show()

+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|fare_amount|trip_distance|
+--------------------+---------------------+------------+------------+---------------+-----------+-------------+
| 2025-01-01 00:27:40|  2025-01-01 00:59:30|         168|          76|              1|       50.5|          0.0|
| 2025-01-01 00:29:37|  2025-01-01 00:29:43|          42|         166|              1|       70.0|         0.03|
| 2025-01-01 00:49:10|  2025-01-01 00:49:14|         143|         143|              4|       85.0|          0.0|
| 2025-01-01 00:34:58|  2025-01-01 01:09:03|          78|         139|              1|       53.5|          0.0|
| 2025-01-01 00:13:47|  2025-01-01 00:14:16|         161|         162|              3|       70.0|         0.06|
| 2025-01-01 00:24:03|  2025-01-01 00:24:06|         161|         161|              3|       79.

## PART C: SPARK STRUCTURED STREAMING

In [29]:
from pyspark.sql.types import *

# C1- Create a streaming source to simulate taxi data

df_streamrate = (
    spark.readStream
    .format("rate")
    .option("rowspersecond",5)
    .load()
)


In [30]:
#Simulate streaming independently and name it like the columns
taxiStream = df_streamrate.select(
    col("timestamp").alias("tpep_pickup_datetime"),
    (col("timestamp") + expr("INTERVAL 10 MINUTES")).alias("tpep_dropoff_datetime"),
    ((col("value") % 265) + 1).cast("int").alias("PULocationID"),
    (((col("value") + 17) % 265) + 1).cast("int").alias("DOLocationID"),
    (col("value") % 10).cast("int").alias("passenger_count"),
    ((col("value") % 50) + 5).cast("double").alias("fare_amount"),
    ((col("value") % 20) + 1).cast("double").alias("trip_distance")
)

In [31]:
#C2- Rolling average fare every 5 minutes
avgFare5min = taxiStream \
.withWatermark("tpep_pickup_datetime", "10 minutes") \
.groupBy(
    window(col("tpep_pickup_datetime"), "5 minutes")
)\
.agg(
    avg("fare_amount").alias("avgFare")
)

In [32]:
#C3- Count Trips per location using windows operation

tripsPerLocation = taxiStream \
.withWatermark("tpep_pickup_datetime", "10 minutes") \
.groupBy(
    window(col("tpep_pickup_datetime"), "5 minutes"),
    col("PULocationID")
) \
.agg(
    count("*").alias("tripCount")
)

In [34]:
#C4- Write Stream
# Console
consoleQuery = avgFare5min.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .start()

26/02/09 19:46:23 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1ddb60fd-92bd-42a2-bb35-7672697de81d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/02/09 19:46:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 13
-------------------------------------------
+------------------------------------------+-----------------+
|window                                    |avgFare          |
+------------------------------------------+-----------------+
|{2026-02-09 19:45:00, 2026-02-09 19:50:00}|29.59452736318408|
+------------------------------------------+-----------------+



-------------------------------------------
Batch: 0
-------------------------------------------
+------+-------+
|window|avgFare|
+------+-------+
+------+-------+



[Stage 85:============> (177 + 5) / 200][Stage 86:>                 (0 + 0) / 4]

In [35]:
#Memory Table
memoryQuery = tripsPerLocation.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("tripsPerLocationtable") \
    .start()


26/02/09 19:46:48 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3db95fb1-3db9-4f36-8cf6-2f5ed6cd6613. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/02/09 19:46:48 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 14
-------------------------------------------
+------------------------------------------+-----------------+
|window                                    |avgFare          |
+------------------------------------------+-----------------+
|{2026-02-09 19:45:00, 2026-02-09 19:50:00}|29.58407079646018|
+------------------------------------------+-----------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-------+
|window                                    |avgFare|
+------------------------------------------+-------+
|{2026-02-09 19:45:00, 2026-02-09 19:50:00}|25.75  |
+------------------------------------------+-------+



[Stage 89:>(77 + 4) / 200][Stage 90:>   (0 + 0) / 4][Stage 92:>   (0 + 0) / 4]4]

In [36]:
# To Parquet- Rolling Average
fileQuery = avgFare5min.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "/home/vboxuser/Big Data Systems Design/Assignment1/tmp/avgFareParquet") \
    .option("checkpointLocation", "/home/vboxuser/Big Data Systems Design/Assignment1/tmp/avgFareParquetCheckpoint") \
    .start()

26/02/09 19:47:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 15
-------------------------------------------
+------------------------------------------+------------------+
|window                                    |avgFare           |
+------------------------------------------+------------------+
|{2026-02-09 19:45:00, 2026-02-09 19:50:00}|29.409506398537477|
+------------------------------------------+------------------+



26/02/09 19:47:24 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/02/09 19:47:24 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/02/09 19:47:24 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/02/09 19:47:24 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.


-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+------------------+
|window                                    |avgFare           |
+------------------------------------------+------------------+
|{2026-02-09 19:45:00, 2026-02-09 19:50:00}|27.714285714285715|
+------------------------------------------+------------------+



26/02/09 19:47:24 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/02/09 19:47:24 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/02/09 19:47:24 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/02/09 19:47:25 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/02/09 19:47:25 WARN HDFSBackedStateStoreProvider: The state for version 16 doesn't exist in loadedMaps. Reading s

...

-------------------------------------------
Batch: 20
-------------------------------------------
+------------------------------------------+------------------+
|window                                    |avgFare           |
+------------------------------------------+------------------+
|{2026-02-09 19:45:00, 2026-02-09 19:50:00}|29.465791292328955|
+------------------------------------------+------------------+



-------------------------------------------
Batch: 7
-------------------------------------------
+------------------------------------------+------------------+
|window                                    |avgFare           |
+------------------------------------------+------------------+
|{2026-02-09 19:45:00, 2026-02-09 19:50:00}|29.240997229916896|
|{2026-02-09 19:50:00, 2026-02-09 19:55:00}|38.5              |
+------------------------------------------+------------------+



-------------------------------------------
Batch: 21
-------------------------------------------
+------------------------------------------+------------------+
|window                                    |avgFare           |
+------------------------------------------+------------------+
|{2026-02-09 19:45:00, 2026-02-09 19:50:00}|29.5              |
|{2026-02-09 19:50:00, 2026-02-09 19:55:00}|28.711267605633804|
+------------------------------------------+------------------+



-------------------------------------------
Batch: 8
-------------------------------------------
+------------------------------------------+------------------+
|window                                    |avgFare           |
+------------------------------------------+------------------+
|{2026-02-09 19:50:00, 2026-02-09 19:55:00}|30.476744186046513|
+------------------------------------------+------------------+



[Stage 145:(85 + 4) / 200][Stage 147:>(0 + 0) / 200][Stage 149:>(0 + 0) / 200]0]

In [ ]:
spark.streams.awaitAnyTermination()

-------------------------------------------
Batch: 22
-------------------------------------------
+------------------------------------------+------------------+
|window                                    |avgFare           |
+------------------------------------------+------------------+
|{2026-02-09 19:50:00, 2026-02-09 19:55:00}|29.333333333333332|
+------------------------------------------+------------------+



-------------------------------------------
Batch: 9
-------------------------------------------
+------------------------------------------+------------------+
|window                                    |avgFare           |
+------------------------------------------+------------------+
|{2026-02-09 19:50:00, 2026-02-09 19:55:00}|29.551136363636363|
+------------------------------------------+------------------+



[Stage 151:(120 + 4) / 200][Stage 153:>(0 + 0) / 200][Stage 155:>(0 + 0) / 200]]

In [ ]:
console_query.stop()
memory_query.stop()
file_query.stop()

## Part D- RDD Operations

In [29]:
#D-1: Map Reduce
PickupFareDF = spark.sql(
    """
    select 
        PULocationID,
        fare_amount,
        passenger_count
    from taxi_trips
    """
)

PickupFareRDD = PickupFareDF.rdd

#filter out all rides that had > 4 passengers
Pcount = PickupFareRDD.filter(lambda r: r.passenger_count > 4)

# For every pickup location show the fare amount
PFare = Pcount.map(lambda r: (r.PULocationID, r.fare_amount))

# sum the fare amount by location
TotalFarePerLocation = PFare.reduceByKey(lambda a, b: a + b)

# display
TotalFarePerLocation.take(10)

[(132, 85960.16999999998),
 (264, 5200.300000000001),
 (68, 14161.299999999994),
 (100, 5907.6),
 (4, 574.1),
 (48, 13930.799999999997),
 (148, 4537.8),
 (12, 215.3),
 (244, 212.0),
 (256, 132.8)]

## 5. Stop Spark Session


In [ ]:
spark.stop()